# 06. Arquitectura RAG (Retrieval-Augmented Generation)
**Objetivo:** Integrar el modelo base (GPT-4o) con datos externos estructurados (`reglas_steam.txt`). Implementamos la cadena completa: **Extracción** (TextLoader), **Fragmentación** (RecursiveCharacterTextSplitter), **Vectorización** (OpenAIEmbeddings + FAISS) y **Recuperación** (retriever).

**Las 4 fases de este módulo:**

1. **Configurar motores:** LLM (para responder) y Embeddings (para traducir texto a vectores).
2. **Ingesta y fragmentación:** leer el `.txt` y cortarlo en fragmentos (*chunks*).
3. **Vectorización:** guardar esos fragmentos en una base de datos vectorial (FAISS).
4. **Recuperación:** buscar la regla exacta y responder el ticket.

## 0. Instalación de dependencias

```bash
pip install -qU langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu
```

## 1. Configuración: motores de Chat y Embeddings

Cargamos las credenciales y levantamos el modelo de chat y el modelo de embeddings (el que convierte el texto en números).

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

# Modelo de chat (temperatura 0.0: respuestas consistentes para soporte)
llm = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.0
)

# Motor de embeddings (modelo pequeño y económico, ideal para esta tarea)
embeddings = OpenAIEmbeddings(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="text-embedding-3-small"
)

print("✅ Motores de Chat y Embeddings configurados.")

## 2. Ingesta y vectorización de la base de conocimiento

Leemos `reglas_steam.txt`, lo fragmentamos en *chunks* y construimos el índice vectorial FAISS que permitirá las búsquedas por similitud.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# 1. Ingesta: leemos la base de conocimiento (mismo directorio que el notebook)
loader = TextLoader("reglas_steam.txt", encoding="utf-8")
documentos = loader.load()

# 2. Fragmentación: cortamos el texto en fragmentos manejables sin perder contexto
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_documents(documentos)

# 3. Vectorización: indexamos los fragmentos en FAISS
vector_db = FAISS.from_documents(chunks, embeddings)

# 4. Recuperador: devuelve los 3 fragmentos más relevantes por consulta
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

print(f"✅ Base de conocimiento indexada: {len(chunks)} fragmentos listos para búsqueda.")

## 3. La cadena RAG y prueba final

Conectamos el *retriever* con el LLM mediante una cadena de recuperación. El *prompt* obliga al bot a responder **solo** con el contexto recuperado, evitando que invente políticas (alucinaciones).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Prompt que ancla la respuesta al contexto oficial recuperado
prompt_rag = ChatPromptTemplate.from_messages([
    ("system",
     "Eres el soporte oficial de Steam. Responde únicamente con la información del "
     "siguiente contexto. Si la respuesta no está en el contexto, di exactamente: "
     "'No tengo información oficial sobre eso'.\n\nContexto:\n{context}"),
    ("human", "{input}")
])

# Ensamblamos la cadena: retriever -> (contexto + pregunta) -> LLM
cadena_documentos = create_stuff_documents_chain(llm, prompt_rag)
cadena_rag = create_retrieval_chain(retriever, cadena_documentos)

# --- Prueba final con un ticket real ---
ticket = "Olvidé mi contraseña y no puedo entrar a mi cuenta, ¿qué hago?"
respuesta = cadena_rag.invoke({"input": ticket})

print(f"🎫 Ticket: {ticket}\n")
print("🤖 Respuesta del bot (RAG):")
print(respuesta["answer"])